raw -> calculated: გაწმენდა, ტიპების გასწორება, ბიზნეს-ლოგიკა.

სკოუპი: მხოლოდ 8000 მ-ზე მაღალი მწვერვალები (16 მწვერვალი).

მიზეზი: 8000 მ-ს ზემოთ („სიკვდილის ზონა") ორგანიზმი ვეღარ აღწევს აკლიმატიზაციას, რომელიცერთგვაროვანი ჯგუფია, სადაც სეზონის, ჟანგბადისა და ოპერატორის შედარება აზრიანია.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS getdata.calculated;


ნაწილი 1 : გაწმენდილი საბაზისო ცხრილები



In [0]:
%sql
--  1.1  CalcPeaks — 8000 მ-ზე მაღალი მწვერვალები
--  16 მწვერვალი 480-დან. სკოუპის ფილტრი აქ ხდება ერთხელ და ქვემოთ ყველა ცხრილი ამ სიას ეყრდნობა.
CREATE OR REPLACE TABLE getdata.calculated.CalcPeaks AS
SELECT
    Pk.peakid AS PeakId,
    Pk.pkname AS PeakName,
    NULLIF(TRIM(Pk.pkname2), '') AS PeakAlternativeName,
    Pk.heightm AS HeightMetres,
    Pk.himal AS MountainRange,
    Pk.region AS Region,
    Pk.pstatus AS ClimbingStatus,
    CAST(Pk.pyear AS INT) AS FirstAscentYear,
    NULLIF(TRIM(Pk.pcountry), '') AS FirstAscentCountry,
    --  სიმაღლის ჯგუფი — ფილტრისთვის
    CASE
        WHEN Pk.heightm >= 8500 THEN '8500+ მ'
        WHEN Pk.heightm >= 8200 THEN '8200–8499 მ'
        ELSE '8000–8199 მ'
    END  AS HeightBand
FROM getdata.raw.raw_peaks AS Pk
WHERE Pk.heightm >= 8000;

COMMENT ON TABLE getdata.calculated.CalcPeaks IS
    'ნეპალის ჰიმალაების 8000 მ-ზე მაღალი მწვერვალები (16 ცალი). სტრიქონი = ერთი მწვერვალი.';


In [0]:
%sql
--  1.2  CalcExpeditions — ექსპედიციები 8000-იანებზე
--  ორი გაწმენდა ხდება აქ:
--
--  (ა) გასაღების კოლიზია. Himalayan Database-ის expid წელს
--      ორნიშნა ციფრით წერს (EVER21101), ამიტომ 1921 და 2021
--      წლის ექსპედიციებს ერთი და იგივე კოდი აქვთ. ასეთი
--      4 კოლიზიაა: EVER21101, EVER22101, EVER24101, KANG10101.
--      ისინი გამოირიცხება, რადგან წევრების მიკუთვნება
--      ცალსახად შეუძლებელია (89 ჩანაწერი, სრულის 0.16%).
--
--  (ბ) season = 'Unknown' -> NULL (1 ექსპედიცია).

CREATE OR REPLACE TABLE getdata.calculated.CalcExpeditions AS
WITH CollidingExpeditionIds AS (
    --  expid, რომელიც ერთზე მეტ ექსპედიციას ეკუთვნის
    SELECT Ex.expid
    FROM getdata.raw.raw_expeditions AS Ex
    GROUP BY Ex.expid
    HAVING COUNT(*) > 1
)
SELECT
    Ex.expid AS ExpeditionId,
    Ex.peakid AS PeakId,
    Ex.year AS ExpeditionYear,
    (Ex.year DIV 10) * 10 AS Decade,
    CASE
        WHEN Ex.year < 1980 THEN 'ექსპლორაცია (1905–1979)'
        WHEN Ex.year < 2000 THEN 'გარდამავალი (1980–1999)'
        ELSE 'კომერციული (2000+)'
    END  AS Era,
    CASE NULLIF(Ex.season, 'Unknown')
        WHEN 'Spring' THEN 'გაზაფხული'
        WHEN 'Summer' THEN 'ზაფხული'
        WHEN 'Autumn' THEN 'შემოდგომა'
        WHEN 'Winter' THEN 'ზამთარი'
    END AS Season,
    CASE NULLIF(Ex.season, 'Unknown')
        WHEN 'Spring' THEN 1
        WHEN 'Summer' THEN 2
        WHEN 'Autumn' THEN 3
        WHEN 'Winter' THEN 4
    END AS SeasonSort,
    NULLIF(TRIM(Ex.nation), '') AS ExpeditionNation,
    NULLIF(TRIM(Ex.agency), '') AS Agency,
    Ex.comrte AS IsCommercialRoute,
    Ex.stdrte AS IsStandardRoute,
    Ex.termreason AS TerminationReason,
    Ex.termreason LIKE 'Success%' AS ReachedSummit,
    Ex.totmembers AS MemberCount,
    Ex.smtmembers AS MemberSummitCount,
    Ex.mdeaths AS MemberDeathCount,
    Ex.tothired AS HiredStaffCount,
    Ex.smthired AS HiredStaffSummitCount,
    Ex.hdeaths AS HiredStaffDeathCount,
    Ex.o2used AS UsedOxygen,
    CAST(Ex.totdays AS INT) AS TotalDays,
    Ex.highpoint AS HighpointMetres
FROM getdata.raw.raw_expeditions AS Ex
INNER JOIN getdata.calculated.CalcPeaks AS Pk
    ON Ex.peakid = Pk.PeakId
WHERE Ex.expid NOT IN (SELECT Cl.expid FROM CollidingExpeditionIds AS Cl);

COMMENT ON TABLE getdata.calculated.CalcExpeditions IS
    'ექსპედიციები 8000 მ-ზე მაღალ მწვერვალებზე, 1905-2024. სტრიქონი = ერთი ექსპედიცია. გამორიცხულია 4 ექსპედიცია გასაღების კოლიზიის გამო.';


In [0]:
%sql
--  1.3  CalcMembers — მთამსვლელები (მთავარი ფაქტების ცხრილი)
-- ეს ცხრილი უმეტესად გამოიყენება დეშბორდის 1-3 გვერდებზე,  
--  exped.year თანმიმდევრულია expeditionის კოდთან, ამიტო ამას ვიღებ ჭეშმარიტ წყაროდ, თან ასაკის გამოთვლასაც ასწორებს და აზუსტებს ექსპედიციის დროს, რადგან იყო ზოგიერთი მონაცემი, მაგალითად: ჰილარი და ტენზინგი ევერესტზე 1953 წლის 29 მაისს ავიდნენ,მათ ჩანაწერებში კი 1984 წერია, რაც უკვე ასაკის გაგებასაც აზიანებს თუ რა დროს ავიდნენ მწვერვალზე
CREATE OR REPLACE TABLE getdata.calculated.CalcMembers AS
WITH MemberBase AS (
    SELECT
        CONCAT(Mb.expid, '-', Mb.membid) AS MemberKey,
        Mb.expid AS ExpeditionId,
        Mb.peakid AS PeakId,
        NULLIF(TRIM(CONCAT_WS(' ', Mb.fname, Mb.lname)), '') AS ClimberName,
        Mb.sex AS Sex,
        CAST(Mb.yob AS INT) AS BirthYear,
        NULLIF(TRIM(Mb.citizen), '') AS Citizenship,
        NULLIF(TRIM(Mb.occupation), '') AS Occupation,
        NULLIF(TRIM(Mb.status), '') AS ExpeditionRole,
        Mb.hired AS IsHiredStaff,
        Mb.sherpa AS IsSherpa,
        Mb.msuccess AS ReachedSummit,
        Mb.mo2used AS UsedOxygen,
        Mb.msolo AS WasSolo,
        Mb.death AS Died,
        Mb.deathtype AS DeathTypeSource,
        Mb.deathclass AS DeathPhaseSource,
        CAST(Mb.deathhgtm AS INT) AS DeathHeightMetres
    FROM getdata.raw.raw_members AS Mb
)
SELECT
    Mb.MemberKey,
    Mb.ExpeditionId,
    Mb.PeakId,
    Pk.PeakName,
    Pk.HeightMetres,
    Pk.HeightBand,
    Ex.ExpeditionYear,
    Ex.Decade,
    Ex.Era,
    Ex.Season,
    Ex.SeasonSort,
    Ex.Agency,
    Ex.IsCommercialRoute,
    Mb.ClimberName,
    Mb.Sex,
    CASE Mb.Sex WHEN 'F' THEN 'ქალი' WHEN 'M' THEN 'კაცი' END AS Gender,
    Mb.BirthYear,
    Mb.Citizenship,
    Mb.Occupation,
    Mb.ExpeditionRole,
    Mb.IsHiredStaff,
    Mb.IsSherpa,
    CASE
        WHEN Mb.IsHiredStaff THEN 'დაქირავებული (შერპა)'
        ELSE 'მთამსვლელი (კლიენტი)'
    END AS StaffType,
    Mb.ReachedSummit,
    Mb.UsedOxygen,
    CASE
        WHEN Mb.UsedOxygen THEN 'ჟანგბადით'
        ELSE  'ჟანგბადის გარეშე'
    END AS OxygenStatus,
    Mb.WasSolo,
    Mb.Died,
    --  ასაკი: მხოლოდ დამაჯერებელ დიაპაზონში
    CASE
        WHEN Ex.ExpeditionYear - Mb.BirthYear BETWEEN 10 AND 90
        THEN Ex.ExpeditionYear - Mb.BirthYear
    END AS AgeAtClimb,
    --  ჯგუფის სახელი ციფრით იწყება ('10–24', და არა '<25'), რომ
    --  დეშბორდზე ანბანური დალაგება ასაკის რიგსაც ემთხვეოდეს.
    CASE
        WHEN Ex.ExpeditionYear - Mb.BirthYear NOT BETWEEN 10 AND 90 THEN NULL
        WHEN Ex.ExpeditionYear - Mb.BirthYear < 25 THEN '10–24'
        WHEN Ex.ExpeditionYear - Mb.BirthYear < 30 THEN '25–29'
        WHEN Ex.ExpeditionYear - Mb.BirthYear < 35 THEN '30–34'
        WHEN Ex.ExpeditionYear - Mb.BirthYear < 40 THEN '35–39'
        WHEN Ex.ExpeditionYear - Mb.BirthYear < 45 THEN '40–44'
        WHEN Ex.ExpeditionYear - Mb.BirthYear < 50 THEN '45–49'
        WHEN Ex.ExpeditionYear - Mb.BirthYear < 55 THEN '50–54'
        WHEN Ex.ExpeditionYear - Mb.BirthYear < 60 THEN '55–59'
        WHEN Ex.ExpeditionYear - Mb.BirthYear < 70 THEN '60–69'
        ELSE '70+'
    END AS AgeBand,
    --  სიკვდილის მიზეზი და ეტაპი ქართულად, დეშბორდისთვის
    CASE Mb.DeathTypeSource
        WHEN 'Avalanche' THEN 'ზვავი'
        WHEN 'Fall' THEN 'ვარდნა'
        WHEN 'AMS (acute mtn sickness)' THEN 'სიმაღლის ავადმყოფობა'
        WHEN 'Illness (non-AMS)' THEN 'ავადმყოფობა'
        WHEN 'Exhaustion' THEN 'გადაღლა'
        WHEN 'Exposure / frostbite' THEN 'გადაცივება / მოყინვა'
        WHEN 'Disappearance (unexplained)' THEN 'უგზო-უკვლოდ დაკარგვა'
        WHEN 'Crevasse' THEN 'ნაპრალში ჩავარდნა'
        WHEN 'Icefall collapse' THEN 'ყინულის ნგრევა'
        WHEN 'Falling rock / ice' THEN 'ქვის / ყინულის ვარდნა'
        WHEN 'Other' THEN 'სხვა'
        WHEN 'Unknown' THEN 'უცნობი'
    END AS DeathType,
    CASE Mb.DeathPhaseSource
        WHEN 'Route preparation' THEN 'მარშრუტის მომზადება'
        WHEN 'Descending from summit bid' THEN 'დაშვება მწვერვალიდან'
        WHEN 'Ascending in summit bid' THEN 'ასვლა მწვერვალზე'
        WHEN 'Death at BC / ABC' THEN 'ბაზაში'
        WHEN 'Expedition evacuation' THEN 'ევაკუაცია'
        WHEN 'Death enroute BC' THEN 'ბაზისკენ მიმავალ გზაზე'
        WHEN 'Other / Unknown' THEN 'სხვა / უცნობი'
    END AS DeathPhase,
    Mb.DeathHeightMetres,
    --  ფსევდო-გასაღები ერთი ადამიანისთვის (იხ. CalcClimberLeaderboard)
    CASE
        WHEN Mb.ClimberName IS NOT NULL AND Mb.BirthYear IS NOT NULL
        THEN CONCAT_WS('|', Mb.ClimberName, CAST(Mb.BirthYear AS STRING), Mb.Citizenship)
    END AS PersonKey
FROM MemberBase AS Mb
INNER JOIN getdata.calculated.CalcExpeditions AS Ex
    ON Mb.ExpeditionId = Ex.ExpeditionId
INNER JOIN getdata.calculated.CalcPeaks AS Pk
    ON Mb.PeakId = Pk.PeakId;

COMMENT ON TABLE getdata.calculated.CalcMembers IS
    'მთავარი ფაქტების ცხრილი: ერთი მთამსვლელი ერთ ექსპედიციაში 8000 მ-ზე მაღალ მწვერვალზე. წელი და სეზონი აღებულია ექსპედიციიდან, რადგან members.myear ძველ ჩანაწერებში დაზიანებულია.';


In [0]:
%sql
--  2.1  CalcWorldPeakStats — 14-ვე 8000-იანი მსოფლიოში
-- ავსებს Himalayan Database-ის ხარვეზს: K2, ნანგა-პარბატი გაშერბრუმები და ბროუდ-პიკი პაკისტანშია და ნეპალის არქივში არ ხვდება.
CREATE OR REPLACE TABLE getdata.calculated.CalcWorldPeakStats AS
SELECT
    TRIM(REGEXP_REPLACE(St.peak_name, '\\(.*\\)', '')) AS PeakName,
    CAST(St.total_ascents AS INT) AS TotalAscents,
    CAST(St.total_deaths AS INT)  AS TotalDeaths,
    CAST(REPLACE(St.deaths_pct_of_ascents, '%', '') AS DOUBLE) AS DeathsPctOfAscents,
    CAST(REPLACE(NULLIF(St.climber_death_rate, '–'), '%', '') AS DOUBLE) AS ClimberDeathRatePct,
    --  არის თუ არა მწვერვალი ნეპალის არქივში (ე.ი. ჩვენს ანალიზში)
    Pk.PeakId IS NOT NULL  AS IsInNepalArchive
FROM getdata.raw.raw_peak_stats_8000ers AS St
LEFT JOIN getdata.calculated.CalcPeaks AS Pk
    ON TRIM(REGEXP_REPLACE(St.peak_name, '\\(.*\\)', '')) = Pk.PeakName;

COMMENT ON TABLE getdata.calculated.CalcWorldPeakStats IS
    'მსოფლიოს 14-ვე 8000-იანის ასვლები და დაღუპულები (Wikipedia). მოიცავს პაკისტანის მწვერვალებსაც, რომლებიც ნეპალის არქივში არ არის.';


In [0]:
%sql
--  2.2  CalcPeakHistory — პირველი და პირველი ზამთრის ასვლა
--  გაწმენდა: '8,849 m(29,032 ft)' -> 8849
--            '29 May 1953'        -> DATE
--  დამატებული მეტრიკა: რამდენი წელი გავიდა პირველ ასვლასა და
--  პირველ ზამთრის ასვლას შორის.

CREATE OR REPLACE TABLE getdata.calculated.CalcPeakHistory AS
WITH ParsedHistory AS (
    SELECT
        TRIM(REGEXP_REPLACE(Fa.peak_name, '\\(.*\\)', '')) AS PeakName,
        CAST(REPLACE(REGEXP_EXTRACT(Fa.height_metres, '([0-9,]+) m', 1), ',', '') AS INT) AS HeightMetres,
        NULLIF(TRIM(Fa.first_ascent_country), '') AS Country,
        NULLIF(TRIM(Fa.first_ascent_date), '') AS FirstAscentDateText,
        NULLIF(TRIM(Fa.first_ascent_summiteers), '') AS FirstAscentTeam,
        NULLIF(TRIM(Fa.first_winter_ascent_date), '') AS FirstWinterAscentDateText,
        NULLIF(TRIM(Fa.first_winter_ascent_summiteers), '') AS FirstWinterAscentTeam,
        CAST(REGEXP_EXTRACT(Fa.first_ascent_date, '([0-9]{4})', 1) AS INT) AS FirstAscentYear,
        CAST(REGEXP_EXTRACT(Fa.first_winter_ascent_date, '([0-9]{4})', 1) AS INT) AS FirstWinterAscentYear
    FROM getdata.raw.raw_peak_first_ascents AS Fa
)
SELECT
    Ph.PeakName,
    Ph.HeightMetres,
    Ph.Country,
    Ph.FirstAscentYear,
    Ph.FirstAscentDateText,
    Ph.FirstAscentTeam,
    Ph.FirstWinterAscentYear,
    Ph.FirstWinterAscentDateText,
    Ph.FirstWinterAscentTeam,
    Ph.FirstWinterAscentYear - Ph.FirstAscentYear AS YearsUntilWinterAscent
FROM ParsedHistory AS Ph;

COMMENT ON TABLE getdata.calculated.CalcPeakHistory IS
    'თითოეული 8000-იანის პირველი და პირველი ზამთრის ასვლა დამპყრობლების სახელებით (Wikipedia).';



In [0]:
%sql
--  2.3  CalcSummiters14 — 14x8000 კლუბი
--  დეშბორდის მე-4 გვერდის მთავარი ცხრილი — ფოტოებით.
--  გაწმენდა:
--    'Mario Vielmo [it]'-> 'Mario Vielmo'(ენის მარკერი)
--    '1970–1986'-> 1970 და 1986(REGEXP)
--    '?–4 October 2024'-> NULL და 2024(დასაწყისი უცნობია)
--    '+1989-10-24T00:00:00Z'-> DATE 1989-10-24(ვიკიდატას ფორმატი)
--  ცოცხალია თუ არა: death_date ცარიელი ნიშნავს, რომ ცოცხალია, ეს მხოლოდ იმ 63 ალპინისტზე ვიცით, ვისაც
--  ვიკიპედიის სტატია აქვს, დანარჩენ 14-ზე მონაცემი არ არსებობს.
CREATE OR REPLACE TABLE getdata.calculated.CalcSummiters14 AS
WITH ParsedSummiters AS (
    SELECT
        TRY_CAST(Su.summit_order AS INT) AS SummitOrder,
        TRY_CAST(NULLIF(TRIM(Su.summit_order_no_oxygen), '') AS INT) AS OxygenFreeOrder,
        TRIM(REGEXP_REPLACE(Su.climber_name, '\\s*\\[[a-z]{2}\\]', '')) AS ClimberName,
        Su.climbing_period AS ClimbingPeriodText,
        --  დასაწყისი უცნობია, თუ ტექსტი '?'-ით იწყება
        CASE
            WHEN Su.climbing_period NOT LIKE '?%'
            THEN CAST(REGEXP_EXTRACT(Su.climbing_period, '([0-9]{4})', 1) AS INT)
        END AS FirstEightThousanderYear,
        CAST(REGEXP_EXTRACT(Su.climbing_period, '.*([0-9]{4})', 1) AS INT) AS CompletionYear,
        CAST(NULLIF(TRIM(Su.born_year), '') AS INT) AS BirthYear,
        CAST(NULLIF(TRIM(Su.age_at_completion), '') AS INT) AS AgeAtCompletion,
        NULLIF(TRIM(Su.nationality), '') AS Nationality,
        NULLIF(TRIM(Su.gender), '')  AS GenderSource,
        NULLIF(TRIM(Su.citizenship), '') AS Citizenship,
        --  ვიკიდატას თარიღი: '+1989-10-24T00:00:00Z'
        TO_DATE(SUBSTR(NULLIF(TRIM(Su.death_date), ''), 2, 10)) AS DeathDate,
        NULLIF(TRIM(Su.death_cause), '') AS DeathCause,
        NULLIF(TRIM(Su.death_place), '') AS DeathPlace,
        NULLIF(TRIM(Su.wiki_url), '') AS WikiUrl,
        NULLIF(TRIM(Su.photo_url), '') AS PhotoUrl
    FROM getdata.raw.raw_summiters_14x8000 AS Su
)
SELECT
    Su.SummitOrder,
    Su.ClimberName,
    Su.Nationality,
    Su.Citizenship,
    CASE Su.GenderSource
        WHEN 'female' THEN 'ქალი'
        WHEN 'male'   THEN 'კაცი'
    END AS Gender,
    Su.BirthYear,
    Su.AgeAtCompletion,
    Su.FirstEightThousanderYear,
    Su.CompletionYear,
    (Su.CompletionYear DIV 10) * 10 AS CompletionDecade,
    Su.CompletionYear - Su.FirstEightThousanderYear AS YearsToComplete,
    Su.OxygenFreeOrder,
    Su.OxygenFreeOrder IS NOT NULL AS IsOxygenFree,
    CASE
        WHEN Su.OxygenFreeOrder IS NOT NULL THEN 'ჟანგბადის გარეშე'
        ELSE 'ჟანგბადით'
    END AS OxygenStatus,
    Su.DeathDate,
    Su.DeathDate IS NOT NULL AS IsDeceased,
    CASE
        WHEN Su.DeathDate IS NOT NULL THEN 'გარდაცვლილი'
        ELSE 'ცოცხალი'
    END  AS LifeStatus,
    Su.DeathCause,
    Su.DeathPlace,
    Su.ClimbingPeriodText,
    Su.WikiUrl,
    Su.PhotoUrl,
    Su.PhotoUrl IS NOT NULL AS HasPhoto
FROM ParsedSummiters AS Su;

COMMENT ON TABLE getdata.calculated.CalcSummiters14 IS
    'ალპინისტები, რომლებმაც 14-ვე 8000-იანი დაიპყრეს (77 ადამიანი). შეიცავს ფოტოს URL-ს და გარდაცვალების მონაცემს ვიკიდატადან.';



ანალიტიკური ცხრილი — ალპინისტების ლიდერბორდი

In [0]:
%sql
--CalcClimberLeaderboard — ვინ ავიდა ყველაზე მეტჯერ

--  მნიშვნელოვანი მეთოდოლოგიური შენიშვნა:Himalayan Database-ს ადამიანის მუდმივი იდენტიფიკატორი არ აქვს membid მხოლოდ ერთი ექსპედიციის ფარგლებშია უნიკალური.ამიტომ ვაშენებთ ფსევდო-გასაღებს: სახელი + დაბადების წელი + მოქალაქეობა.

--  მარტო სახელი საკმარისი არ არის: ნეპალურ სახელებში თანხვედრა ხშირია და მარტო სახელით დაჯგუფება 'Lhakpa Nuru Sherpa'-ს 123 ასვლას მიაწერდა 1980-2024 წლებში, და ეს შესაძლოა რამდენიმე სხვადასხვა ადამიანიც იყოს.

--  დაბადების წლის დამატებით შედეგი რეალობას ემთხვევა: Kami Rita (Topke) Sherpa, დაბ. 1970  ევერესტზე 27-ჯერ, რაც მსოფლიო რეკორდია და გარე წყაროებით დასტურდება.
--  მაინც შესაძლებელია ცალკეული შეცდომა, ამიტომ ცხრილი რეპორტში წარმოდგენილია როგორც შეფასება, არა როგორც ოფიციალური რეესტრი.

CREATE OR REPLACE TABLE getdata.calculated.CalcClimberLeaderboard AS
SELECT
    Mb.PersonKey,
    MAX(Mb.ClimberName) AS ClimberName,
    MAX(Mb.BirthYear) AS BirthYear,
    MAX(Mb.Citizenship) AS Citizenship,
    MAX(Mb.IsSherpa)  AS IsSherpa,
    MAX(Mb.IsHiredStaff) AS IsHiredStaff,
    COUNT(*) AS SummitCount,
    COUNT(DISTINCT Mb.PeakId) AS DistinctPeakCount,
    SUM(CASE WHEN Mb.PeakId = 'EVER' THEN 1 ELSE 0 END) AS EverestSummitCount,
    SUM(CASE WHEN NOT Mb.UsedOxygen THEN 1 ELSE 0 END) AS OxygenFreeSummitCount,
    MIN(Mb.ExpeditionYear) AS FirstSummitYear,
    MAX(Mb.ExpeditionYear) AS LastSummitYear,
    MAX(Mb.ExpeditionYear) - MIN(Mb.ExpeditionYear) + 1 AS CareerYears,
    MIN(Mb.AgeAtClimb) AS YoungestSummitAge,
    MAX(Mb.AgeAtClimb) AS OldestSummitAge
FROM getdata.calculated.CalcMembers AS Mb
WHERE Mb.ReachedSummit
  AND Mb.PersonKey IS NOT NULL
GROUP BY Mb.PersonKey
HAVING COUNT(*) >= 2;

COMMENT ON TABLE getdata.calculated.CalcClimberLeaderboard IS
    'ალპინისტები 2+ ასვლით 8000-იანებზე. ადამიანი იდენტიფიცირდება ფსევდო-გასაღებით (სახელი + დაბადების წელი + მოქალაქეობა), რადგან წყაროს მუდმივი პირადი კოდი არ აქვს.';
